In [6]:
from gorgo import infer, condition, draw_from, flip, keep_deterministic, mem, factor
from gorgo.hashable import hashabledict
from gorgo.distributions.builtin_dists import Uniform, Beta
from gorgo.distributions import Mixture
from gorgo.distributions.random import RandomNumberGenerator

from model import Utterance, Instance, Kind, meaning, literal_listener, speaker, pragmatic_listener

import numpy as np
from gorgo.inference import MaximumMarginalAPosteriori
import pickle

import pandas as pd

# Load model fitted output

from model-study6-1-fit.ipynb

In [7]:
########
# Load coherence distributions from each model for each condition
########

# load pragmatic listener coherence outputs
with open("scratch/study 6/dist_coherence_generic_prag.pkl", "rb") as f: 
    dist_coherence_generic_prag = pickle.load(f)

with open("scratch/study 6/dist_coherence_specific_prag.pkl", "rb") as f:
    dist_coherence_specific_prag = pickle.load(f)

with open("scratch/study 6/dist_coherence_baseline_prag.pkl", "rb") as f:
    dist_coherence_baseline_prag = pickle.load(f)

# load literal listener coherence outputs
with open("scratch/study 6/dist_coherence_generic_lit.pkl", "rb") as f: 
    dist_coherence_generic_lit = pickle.load(f)

with open("scratch/study 6/dist_coherence_specific_lit.pkl", "rb") as f:
    dist_coherence_specific_lit = pickle.load(f)

with open("scratch/study 6/dist_coherence_baseline_lit.pkl", "rb") as f:
    dist_coherence_baseline_lit = pickle.load(f)
    
# load base model coherence outputs
with open("scratch/study 6/dist_coherence_generic_base.pkl", "rb") as f: 
    dist_coherence_generic_base = pickle.load(f)

with open("scratch/study 6/dist_coherence_specific_base.pkl", "rb") as f:
    dist_coherence_specific_base = pickle.load(f)

with open("scratch/study 6/dist_coherence_baseline_base.pkl", "rb") as f:
    dist_coherence_baseline_base = pickle.load(f)

    

########
# Load MLE results for each feature and condition from each model (from linking function)
########
mle_per_feature_prag = pd.read_csv('scratch/study 6/mle_per_feature_prag.csv')
mle_per_feature_lit = pd.read_csv('scratch/study 6/mle_per_feature_lit.csv')
mle_per_feature_base = pd.read_csv('scratch/study 6/mle_per_feature_base.csv')

# Define helper functions

In [8]:
# put row-wise loop into separate function to speed up runtime
def observe_response(
    condition, 
    prevalence,
    feature_kind_linked_prevalence, 
    feature_not_kind_linked_prevalence, 
    model,
    simulate_response = False
):
    if model == "pragmatic": 
        # get the coherence distribution for the condition
        if condition == "generic":
            dist_coherence = dist_coherence_generic_prag

        elif condition == "specific":
            dist_coherence = dist_coherence_specific_prag
            
        elif condition == "baseline":
            dist_coherence = dist_coherence_baseline_prag
        else:
            raise ValueError("Condition must be 'generic', 'specific', or 'baseline'")

    elif model == "literal":
        # get the coherence distribution for the condition
        if condition == "generic":
            dist_coherence = dist_coherence_generic_lit

        elif condition == "specific":
            dist_coherence = dist_coherence_specific_lit
            
        elif condition == "baseline":
            dist_coherence = dist_coherence_baseline_lit
        else:
            raise ValueError("Condition must be 'generic', 'specific', or 'baseline'")
            
    elif model == "base":
        # get the coherence distribution for the condition
        if condition == "generic":
            dist_coherence = dist_coherence_generic_base

        elif condition == "specific":
            dist_coherence = dist_coherence_specific_base
            
        elif condition == "baseline":
            dist_coherence = dist_coherence_baseline_base
        else:
            raise ValueError("Condition must be 'generic', 'specific', or 'baseline'")
    else:
        raise ValueError("Model must be 'pragmatic', 'literal', or 'base'")

    # flip on kind-linked status using a sampled coherence from the pragmatic listener
    coherence = dist_coherence.sample() # this might just be the EV of coherence
    feature_is_kind_linked = flip(coherence)
    
    # get response distribution
    if feature_is_kind_linked:
        response_dist = feature_kind_linked_prevalence
    else:
        response_dist = feature_not_kind_linked_prevalence
    
    if simulate_response:
        # simulate a response
        response = response_dist.sample()
        return response
    else:
        # observe actual response (this reweights the distribution)
        response_dist.observe(prevalence)



##### Convert gorgo dist to df for export
@keep_deterministic
def dist_to_df(dist) -> pd.DataFrame:
    
    # validate input
    if not hasattr(dist, 'support') or not hasattr(dist, 'probabilities'):
        raise AttributeError("Distribution must have support and probabilities")
    
    # create dataframe based on support type
    if isinstance(dist.support[0], float): 
        df_dist = pd.DataFrame({
            'Element': dist.support,
            'Probability': dist.probabilities
        })
    else: 
        # for complex support, expand into multiple columns
        df_dist = pd.DataFrame(dist.support)
        df_dist['Probability'] = dist.probabilities
    
    return df_dist

# Set up Study 6 structure

In [9]:
# conditions
conditions = ("generic", "baseline", "specific")

# training trials per condition
generic_condition = (
    (Utterance("Zarpies", "love to eat flowers"), Instance("Zarpie", ("love to eat flowers", ))),
    (Utterance("Zarpies", "have stripes in their hair"), Instance("Zarpie", ("have stripes in their hair", ))),
    (Utterance("Zarpies", "can bounce a ball on their heads"), Instance("Zarpie", ("can bounce a ball on their heads", ))),
    (Utterance("Zarpies", "like to sing"), Instance("Zarpie", ("like to sing", ))),
    (Utterance("Zarpies", "climb tall fences"), Instance("Zarpie", ("climb tall fences", ))),
    (Utterance("Zarpies", "flap their arms when they are happy"), Instance("Zarpie", ("flap their arms when they are happy", ))),
    (Utterance("Zarpies", "have freckles on their feet"), Instance("Zarpie", ("have freckles on their feet", ))),
    (Utterance("Zarpies", "hop over puddles"), Instance("Zarpie", ("hop over puddles", ))),
    (Utterance("Zarpies", "really don't like walking in the mud"), Instance("Zarpie", ("really don't like walking in the mud", ))),
    (Utterance("Zarpies", "draw stars on their knees"), Instance("Zarpie", ("draw stars on their knees", ))),
    (Utterance("Zarpies", "can flip in the air"), Instance("Zarpie", ("can flip in the air", ))),
    (Utterance("Zarpies", "are scared of ladybugs"), Instance("Zarpie", ("are scared of ladybugs", ))),
    (Utterance("Zarpies", "really don't like ice cream"), Instance("Zarpie", ("really don't like ice cream", ))),
    (Utterance("Zarpies", "chase shadows"), Instance("Zarpie", ("chase shadows", ))),
    (Utterance("Zarpies", "babies are wrapped in orange blankets"), Instance("Zarpie", ("babies are wrapped in orange blankets", ))),
    (Utterance("Zarpies", "sleep in tall trees"), Instance("Zarpie", ("sleep in tall trees", )))
)

specific_condition = (
    (Utterance("This Zarpie", "love to eat flowers"), Instance("Zarpie", ("love to eat flowers", ))),
    (Utterance("This Zarpie", "have stripes in their hair"), Instance("Zarpie", ("have stripes in their hair", ))),
    (Utterance("This Zarpie", "can bounce a ball on their heads"), Instance("Zarpie", ("can bounce a ball on their heads", ))),
    (Utterance("This Zarpie", "like to sing"), Instance("Zarpie", ("like to sing", ))),
    (Utterance("This Zarpie", "climb tall fences"), Instance("Zarpie", ("climb tall fences", ))),
    (Utterance("This Zarpie", "flap their arms when they are happy"), Instance("Zarpie", ("flap their arms when they are happy", ))),
    (Utterance("This Zarpie", "have freckles on their feet"), Instance("Zarpie", ("have freckles on their feet", ))),
    (Utterance("This Zarpie", "hop over puddles"), Instance("Zarpie", ("hop over puddles", ))),
    (Utterance("This Zarpie", "really don't like walking in the mud"), Instance("Zarpie", ("really don't like walking in the mud", ))),
    (Utterance("This Zarpie", "draw stars on their knees"), Instance("Zarpie", ("draw stars on their knees", ))),
    (Utterance("This Zarpie", "can flip in the air"), Instance("Zarpie", ("can flip in the air", ))),
    (Utterance("This Zarpie", "are scared of ladybugs"), Instance("Zarpie", ("are scared of ladybugs", ))),
    (Utterance("This Zarpie", "really don't like ice cream"), Instance("Zarpie", ("really don't like ice cream", ))),
    (Utterance("This Zarpie", "chase shadows"), Instance("Zarpie", ("chase shadows", ))),
    (Utterance("This Zarpie", "babies are wrapped in orange blankets"), Instance("Zarpie", ("babies are wrapped in orange blankets", ))),
    (Utterance("This Zarpie", "sleep in tall trees"), Instance("Zarpie", ("sleep in tall trees", )))
)

baseline_condition = ()

# Simulate prevalence responses from each model

In [10]:
##### Define helper functions

# simulate responses from a given model
def simulate_responses_from_model(model, model_mle_per_feature, 
                                  conditions = ["generic", "baseline", "specific"], 
                                  simulations = 300):
    data_simulated_feature = []
        
    # for each condition...
    for condition in conditions:
        # run 300 simulations (=~3x actual participants):
        for simulation in range(1, simulations + 1):
            # for each of 16 features...
            for i, row in model_mle_per_feature.iterrows():
                # construct beta distributions for the given feature
                feature_kind_linked_prevalence = Beta(row['feature_kind_linked_prevalence_alpha'], 
                                                      row['feature_kind_linked_prevalence_beta'])
                feature_not_kind_linked_prevalence = Beta(row['feature_not_kind_linked_prevalence_alpha'], 
                                                          row['feature_not_kind_linked_prevalence_beta'])
            
                # simulate a response
                response = observe_response(
                    condition = condition,
                    prevalence = None, # irrelevant for simulating response
                    feature_kind_linked_prevalence = feature_kind_linked_prevalence,
                    feature_not_kind_linked_prevalence = feature_not_kind_linked_prevalence,
                    simulate_response = True,
                    model = model
                )
                
                # record results
                data_simulated_feature.append({
                    'condition': condition,
                    'simulation': simulation,
                    'feature': row['feature'],
                    'prevalence': response
                })
    return data_simulated_feature

In [11]:
##### Simulate responses from each model

# get simulated data from pragmatic listener
data_prag_feature = simulate_responses_from_model(model = "pragmatic", 
                                                  model_mle_per_feature = mle_per_feature_prag)
df_prag_feature = pd.DataFrame(data_prag_feature)

# match real data by rounding, clipping prevalence to .01-.99, ordering condition and features
df_prag_feature['prevalence'] = df_prag_feature['prevalence'].round(2)
df_prag_feature["prevalence"] = df_prag_feature["prevalence"].replace(1, .99)
df_prag_feature["prevalence"] = df_prag_feature["prevalence"].replace(0, .01)
df_prag_feature["condition"] = df_prag_feature["condition"].astype(
    pd.CategoricalDtype(categories=["generic", "baseline", "specific"], ordered=True))
df_prag_feature = df_prag_feature.sort_values(by='feature') 


# get simulated data from literal listener
data_lit_feature = simulate_responses_from_model(model = "literal", 
                                                  model_mle_per_feature = mle_per_feature_lit)
df_lit_feature = pd.DataFrame(data_lit_feature)

# match real data by rounding, clipping prevalence to .01-.99, ordering condition and features
df_lit_feature['prevalence'] = df_lit_feature['prevalence'].round(2)
df_lit_feature["prevalence"] = df_lit_feature["prevalence"].replace(1, .99)
df_lit_feature["prevalence"] = df_lit_feature["prevalence"].replace(0, .01)
df_lit_feature["condition"] = df_lit_feature["condition"].astype(
    pd.CategoricalDtype(categories=["generic", "baseline", "specific"], ordered=True))
df_lit_feature = df_lit_feature.sort_values(by='feature')


# get simulated data from base model
data_base_feature = simulate_responses_from_model(model = "base", 
                                                  model_mle_per_feature = mle_per_feature_base)
df_base_feature = pd.DataFrame(data_base_feature)

# match real data by rounding, clipping prevalence to .01-.99, ordering condition and features
df_base_feature['prevalence'] = df_base_feature['prevalence'].round(2)
df_base_feature["prevalence"] = df_base_feature["prevalence"].replace(1, .99)
df_base_feature["prevalence"] = df_base_feature["prevalence"].replace(0, .01)
df_base_feature["condition"] = df_base_feature["condition"].astype(
    pd.CategoricalDtype(categories=["generic", "baseline", "specific"], ordered=True))
df_base_feature = df_base_feature.sort_values(by='feature')

In [85]:
# export simulated data to csv
df_prag_feature.to_csv("scratch/study 6/df_prag_feature.csv", index=False)
df_lit_feature.to_csv("scratch/study 6/df_lit_feature.csv", index=False)
df_base_feature.to_csv("scratch/study 6/df_base_feature.csv", index=False)